# cadence 学習環境セットアップ【Colab・最初に1回】

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slp-hu/Style-Bert-VITS2/blob/layer-b-cadence-seq/colab/cadence_train_setup_colab.ipynb)

学習側 Drive に **fork clone（= 学習ルート）** を作り、底モデル等を配置する。学習ノートより先に実行。
**コーパス非依存**（CV / CSJ どちらでもこのノートは共通・1回だけでよい）。

| 区分 | 内容 | 永続性 |
|---|---|---|
| **1回だけ**（§1,§3,§4） | fork clone / `initialize.py`（bert・slm・**pretrained_jp_extra**）/ sanity | Drive に永続 |
| **セッション毎**（§2） | 依存インストール | Colab VM 揮発（毎回実行） |

- このノートは **CPU ランタイムで可**（GPU 向けの torch 入替や shim は学習ノート
  `cadence_train_colab.ipynb` の §2/§3 が GPU を判定して自動で行う）。
- 底モデル（warm-start 元）= `pretrained_jp_extra/{G_0, D_0, WD_0}.safetensors`
  （HF `litagin/Style-Bert-VITS2-2.0-base-JP-Extra` から initialize.py が自動取得）。
  学習済み重みはこの底モデルのライセンス（AGPL-3.0）に従属する。
  **CSJ 由来の重みはそれとは別に二次配布の制約がかかる**（README のライセンス節を参照）。
- Drive 容量目安: clone＋bert/slm/pretrained ≈ 数 GB。無料枠(15GB)は要注意
  （データセット本体は学習ノートが**ローカル /content に展開**するので Drive を消費しない）。


In [ ]:
# ===== §1 Drive マウント & fork clone（1回だけ）=====
from google.colab import drive
from pathlib import Path
import subprocess, os

def _drive_alive():
    try:
        next(iter(os.listdir("/content/drive/MyDrive")), None)
        return True
    except OSError:
        return False

drive.mount('/content/drive')
if not _drive_alive():
    print("★Drive マウントが切れている → 強制再マウント")
    subprocess.run(["fusermount", "-u", "/content/drive"], capture_output=True)
    drive.mount('/content/drive', force_remount=True)
    assert _drive_alive(), "★再マウント失敗 → ランタイム再起動してやり直す"

FORK_URL = 'https://github.com/slp-hu/Style-Bert-VITS2.git'   # 公開 fork（匿名 clone 可）
BRANCH   = 'layer-b-cadence-seq'
PARENT   = Path('/content/drive/MyDrive')           # clone を置く親（任意）
BASE     = PARENT / 'Style-Bert-VITS2'              # 学習ルート = fork clone 直下（学習/評価ノートの DRIVE_BASE と一致させる）

if not BASE.exists():
    print('clone:', FORK_URL, '->', BASE)
    subprocess.run(['git', 'clone', '-b', BRANCH, FORK_URL, str(BASE)], check=True)
else:
    print('既存 clone を使用:', BASE)
    try:
        _p = subprocess.run(['git', '-C', str(BASE), 'pull', '--ff-only'],
                            capture_output=True, text=True, timeout=180)
        print('git pull:', (_p.stdout.strip().splitlines() or ['?'])[-1] if _p.returncode == 0
              else '★失敗（続行）: ' + (_p.stderr or '').strip()[-200:])
    except Exception as e:
        print('★git pull 例外（続行）:', e)

os.chdir(BASE)
br = subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'], capture_output=True, text=True).stdout.strip()
print('branch:', br)
assert br == BRANCH, f"★branch が {BRANCH} でない: {br} → git checkout {BRANCH}"


In [ ]:
# ===== §2 依存インストール（セッション毎に実行）=====
# 【av ビルド失敗の恒久対策】
#   requirements.txt の faster-whisper==0.10.1 が av==10.* をソースビルドしようとし、
#   Colab(Python 3.12)で wheel を作れず失敗する。faster-whisper / av は文字起こし用途で
#   cv_r1 の用途（initialize / bert_gen / style_gen / train / eval）には不要 → 行を除外して入れる。
# 【HF スタック固定】
#   requirements は transformers 未ピン → Colab 既定の transformers 5.x が torch>=2.4 を要求して
#   torch を無効化し、後段の bert_gen が "AutoModelForMaskedLM requires PyTorch" で落ちる。
#   → 動作確認済みの組合せに固定する（学習/評価ノートの §2 と同一）。
# 【torch について】
#   このノートでは requirements の pin（torch<2.4 = 2.3.1）のまま。GPU での torch の扱い
#   （Blackwell = sm_120 なら 2.11+cu128 へ入替 + shim）は学習ノート §2/§3 が自動で行う。
import subprocess, sys, re
from pathlib import Path

req = Path("requirements.txt")
req_train = Path("requirements_no_whisper.txt")
_drop = re.compile(r"^(faster-whisper|av)==")   # ビルドで詰まるパッケージが増えたらここに追加
kept = [l for l in req.read_text(encoding="utf-8").splitlines() if not _drop.match(l.strip())]
req_train.write_text("\n".join(kept) + "\n", encoding="utf-8")
print(f"除外後 requirements: {req_train} ({len(kept)} 行)")

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_train)],
                   capture_output=True, text=True)
print(r.stdout[-2000:] or "(quiet)")
if r.returncode != 0:
    print(r.stderr[-4000:]); raise SystemExit("pip 失敗")

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.41.2", "huggingface_hub==0.23.5", "tokenizers<0.20",
                    "pytorch-lightning==2.2.5", "torchmetrics<1.5", "pyannote.audio==3.1.1",
                    "scipy==1.13.1", "numpy==1.26.4"], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-4000:]); raise SystemExit("HF スタック固定 失敗")

# 検証は別プロセスで（transformers から torch が見えているかまで確認）
v = subprocess.run([sys.executable, "-c",
    "import torch;from transformers.utils import is_torch_available;"
    "print('torch', torch.__version__, '| transformers_sees_torch', is_torch_available())"],
    capture_output=True, text=True)
print(v.stdout.strip() or v.stderr[-800:])
assert "transformers_sees_torch True" in v.stdout, "★transformers が torch を認識していない"
import importlib
for m in ["pyopenjtalk_dict", "torchaudio", "numpy", "scipy"]:
    print("  import OK:", m) if importlib.util.find_spec(m) else print("  ★import NG:", m)


In [ ]:
# ===== §3 initialize（1回だけ・Drive に永続）=====
# bert / slm(wavlm) / pretrained / pretrained_jp_extra(底モデル) を clone 内へダウンロード
import subprocess, sys
r = subprocess.run([sys.executable, 'initialize.py', '--skip_default_models'],
                   capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-1000:])
assert r.returncode == 0, 'initialize.py 失敗'


In [ ]:
# ===== §4 sanity（毎回実行して良い）=====
import hashlib
from pathlib import Path

need = [
    'pretrained_jp_extra/G_0.safetensors',   # 底モデル（warm-start 元）
    'pretrained_jp_extra/D_0.safetensors',
    'pretrained_jp_extra/WD_0.safetensors',
    'slm/wavlm-base-plus/pytorch_model.bin',
    'configs/config_jp_extra.json',
    'configs/paths.yml',
]
ng = 0
for f in need:
    p = BASE / f
    ok = p.exists()
    print(('OK ' if ok else '★NG') , f, f'({p.stat().st_size/2**20:.0f} MiB)' if ok else '')
    ng += (not ok)
assert ng == 0, '不足あり: initialize.py の出力を確認'

h = hashlib.sha256((BASE/'configs'/'config_jp_extra.json').read_bytes()).hexdigest()
print('config_jp_extra.json sha256 一致:', h == '69d907db9f38bf05f52d117814d38808635a4975de38fae5c775f4ff47d42187')

import yaml
py = yaml.safe_load(open(BASE/'configs'/'paths.yml', encoding='utf-8'))
print('paths.yml dataset_root:', py.get('dataset_root'), '（Data のままで良い: データは Data/cv_r1 に展開される）')


## 次の手順

1. **`cadence_train_colab.ipynb`**（GPU ランタイム）を実行する。§1.2 で `CORPUS`（`cv_r1` / `csj_r1`）を
   選ぶ。学習ノートが以下を一括で面倒を見る:
   - GPU 自動判定（標準 GPU = torch 2.3.1 のまま / **Blackwell** = torch 2.11+cu128 へ入替 →
     ランタイム再起動 → sitecustomize shim）
   - **学習バンドルのローカル `/content` 展開**（Drive 直読みの I/O 律速を回避。
     checkpoint / model_assets は Drive へ symlink して永続化）
     - `cv_r1`: Zenodo から自動 DL
     - `csj_r1`: **自動 DL しない**。`DRIVE_BASE/corpus_in/csj_r1.tar.gz` を各自で置く
       （作り方は `docs/TRAIN_BUNDLE_SPEC.md`）
   - `default_style.py` の cadseq パッチ・底モデル配置・検証ゲート・bert_gen / style_gen / 学習
2. 学習完走後は **`cv_r1_eval_speaker.ipynb`**（弾き分け / UTMOS / virtual 話者）へ。
   ※ eval / synth / demo ノートは**まだ cv_r1 固定**（CORPUS 対応は次段）。

参考: 学習ノート §6〜§8 が実行する CLI（cwd = fork 直下、パスはすべて cwd 起点の相対）:
```
python bert_gen.py           -c Data/{CORPUS}/config.json
python style_gen.py          -c Data/{CORPUS}/config.json
python train_ms_jp_extra.py  -c Data/{CORPUS}/config.json -m Data/{CORPUS}
```
- ★`-m {CORPUS}` は誤り（checkpoint がリポジトリ直下の別ツリーに落ちる）。正しくは `-m Data/{CORPUS}`。
- `preprocess_text` は**走らせない**（esd 配置済み・spk2id 再生成の恐れ）。
